# M1 Notebook 15 — Expectation, Variance, Covariance, and Concentration

**Notebook ID:** M1_N15  
**Status:** Runnable first edition  
**Random seed:** 42

> Expectation summarizes location, variance summarizes dispersion, covariance summarizes joint movement, and concentration bounds quantify how unlikely large deviations can be.


## 1. Learning objectives

1. Compute expectation and variance for discrete random variables.
2. Interpret covariance and correlation.
3. Construct covariance matrices.
4. Demonstrate linearity of expectation.
5. Compare covariance with causation.
6. Apply Markov, Chebyshev, and Hoeffding bounds.
7. Connect moments and concentration to risk and Decision Intelligence.


In [ ]:
from srai_math.utils import environment_info, set_seed
from srai_math.probability import (
    chebyshev_bound, correlation, covariance, covariance_matrix,
    discrete_expectation, discrete_variance, hoeffding_bound,
    markov_bound, running_mean, running_variance,
)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
set_seed(42)
environment_info()


## 2. Expectation

For a discrete random variable,

\[
\mathbb E[X]
=
\sum_x xP(X=x).
\]

Expectation is a probability-weighted average, not necessarily an observable outcome.


In [ ]:
values = np.array([0.0, 1.0, 2.0])
probabilities = np.array([0.25, 0.50, 0.25])

mean = discrete_expectation(values, probabilities)
assert np.isclose(mean, 1.0)
mean


## 3. Variance

\[
\operatorname{Var}(X)
=
\mathbb E[(X-\mu)^2]
=
\mathbb E[X^2]-\mu^2.
\]


In [ ]:
variance = discrete_variance(values, probabilities)
second_moment = discrete_expectation(values**2, probabilities)

assert np.isclose(variance, second_moment - mean**2)
variance


## 4. Linearity of expectation

For constants \(a,b\),

\[
\mathbb E[aX+bY]
=
a\mathbb E[X]+b\mathbb E[Y].
\]

Independence is not required.


In [ ]:
x_values = np.array([1.0, 2.0, 3.0])
y_values = np.array([4.0, 2.0, 0.0])
p = np.array([0.2, 0.5, 0.3])

lhs = discrete_expectation(2*x_values - y_values, p)
rhs = (
    2*discrete_expectation(x_values, p)
    - discrete_expectation(y_values, p)
)

assert np.isclose(lhs, rhs)
lhs, rhs


## 5. Covariance

\[
\operatorname{Cov}(X,Y)
=
\mathbb E[(X-\mu_X)(Y-\mu_Y)].
\]

Positive covariance indicates joint movement in the same direction; negative covariance indicates opposite movement.


In [ ]:
x = np.array([1, 2, 3, 4, 5], dtype=float)
y = np.array([2, 4, 5, 8, 10], dtype=float)

cov_xy = covariance(x, y)
corr_xy = correlation(x, y)

{
    "covariance": cov_xy,
    "correlation": corr_xy,
}


## 6. Correlation

\[
\rho_{XY}
=
\frac{\operatorname{Cov}(X,Y)}
{\sigma_X\sigma_Y}.
\]

Correlation is dimensionless and lies between \(-1\) and \(1\).


## 7. Scaling experiment

In [ ]:
scaled_y = 100 * y

comparison = {
    "covariance_original": covariance(x, y),
    "covariance_scaled": covariance(x, scaled_y),
    "correlation_original": correlation(x, y),
    "correlation_scaled": correlation(x, scaled_y),
}
comparison


Covariance changes with scale; correlation does not.


## 8. Covariance is not causation

In [ ]:
rng = np.random.default_rng(42)
temperature = rng.normal(30, 4, 500)
ice_cream_sales = 50 + 3*temperature + rng.normal(0, 8, 500)
electricity_demand = 100 + 4*temperature + rng.normal(0, 10, 500)

spurious_frame = pd.DataFrame({
    "Ice cream sales": ice_cream_sales,
    "Electricity demand": electricity_demand,
    "Temperature": temperature,
})

spurious_frame.corr()


Ice-cream sales and electricity demand are correlated because both respond to temperature. Their correlation does not mean one causes the other.


## 9. Covariance matrix

For a random vector \(\mathbf X\),

\[
\Sigma
=
\mathbb E[
(\mathbf X-\boldsymbol\mu)
(\mathbf X-\boldsymbol\mu)^\top
].
\]


In [ ]:
X = spurious_frame.to_numpy()
Sigma = covariance_matrix(X)

covariance_frame = pd.DataFrame(
    Sigma,
    index=spurious_frame.columns,
    columns=spurious_frame.columns,
)
covariance_frame


In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
image = ax.imshow(covariance_frame.to_numpy())
ax.set_xticks(range(len(covariance_frame.columns)), covariance_frame.columns, rotation=45, ha="right")
ax.set_yticks(range(len(covariance_frame.index)), covariance_frame.index)
ax.set_title("Covariance Matrix")
fig.colorbar(image, ax=ax)
plt.tight_layout()
plt.show()


## 10. Running means and variances

Repeated observations stabilize summary estimates over time.


In [ ]:
samples = rng.normal(loc=10.0, scale=2.0, size=5000)
mean_path = running_mean(samples)
variance_path = running_variance(samples)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(mean_path, label="Running mean")
ax.axhline(10.0, linestyle="--", label="True mean")
ax.set_xlabel("Sample size")
ax.set_ylabel("Mean")
ax.set_title("Convergence of the Running Mean")
ax.legend()
plt.show()


## 11. Markov inequality

For a nonnegative random variable \(X\),

\[
P(X\ge a)
\le
\frac{\mathbb E[X]}{a}.
\]


In [ ]:
markov = markov_bound(
    expectation=2.0,
    threshold=5.0,
)
markov


## 12. Chebyshev inequality

For any random variable with finite variance,

\[
P(|X-\mu|\ge k)
\le
\frac{\sigma^2}{k^2}.
\]


In [ ]:
chebyshev = chebyshev_bound(
    variance=4.0,
    deviation=3.0,
)
chebyshev


## 13. Hoeffding inequality

For independent bounded observations \(X_i\in[a,b]\),

\[
P(|\bar X-\mathbb E[X]|\ge\varepsilon)
\le
2\exp\left(
-\frac{2n\varepsilon^2}{(b-a)^2}
\right).
\]


In [ ]:
sample_sizes = np.array([50, 100, 250, 500, 1000])
bounds = [
    hoeffding_bound(
        epsilon=0.05,
        n=int(n),
        lower=0.0,
        upper=1.0,
    )
    for n in sample_sizes
]

pd.DataFrame({
    "sample_size": sample_sizes,
    "Hoeffding_bound": bounds,
})


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.semilogy(sample_sizes, bounds, marker="o")
ax.set_xlabel("Sample size")
ax.set_ylabel("Upper probability bound")
ax.set_title("Hoeffding Concentration with Increasing Sample Size")
plt.show()


## 14. Empirical concentration experiment

In [ ]:
trials = 5000
n = 200
epsilon = 0.05

sample_means = rng.binomial(
    1,
    0.5,
    size=(trials, n),
).mean(axis=1)

empirical_tail = np.mean(
    np.abs(sample_means - 0.5) >= epsilon
)
hoeffding_tail = hoeffding_bound(
    epsilon,
    n,
    lower=0.0,
    upper=1.0,
)

{
    "empirical_tail_probability": empirical_tail,
    "Hoeffding_upper_bound": hoeffding_tail,
}


Concentration inequalities are usually conservative. Their value is that they offer guarantees under stated assumptions.


## 15. Statistics interpretation

Moments summarize distributions and dependence. Covariance matrices support regression, PCA, multivariate analysis, and uncertainty propagation.


## 16. AI interpretation

Expectation and variance appear in:

- loss functions;
- stochastic gradients;
- uncertainty estimation;
- batch normalization;
- Bayesian models;
- exploration strategies;
- generalization bounds.


## 17. Decision Intelligence case — Portfolio of sector risks

Suppose agriculture, energy, and transport losses are jointly uncertain. Total risk depends on both individual variances and cross-sector covariance.


In [ ]:
losses = pd.DataFrame({
    "Agriculture": rng.normal(50, 12, 2000),
    "Energy": rng.normal(40, 9, 2000),
    "Transport": rng.normal(30, 7, 2000),
})

common_shock = rng.normal(0, 8, 2000)
losses["Agriculture"] += common_shock
losses["Energy"] += 0.8*common_shock
losses["Transport"] += 0.5*common_shock

weights = np.array([0.4, 0.35, 0.25])
sector_covariance = covariance_matrix(losses.to_numpy())
portfolio_variance = weights @ sector_covariance @ weights
portfolio_std = np.sqrt(portfolio_variance)

{
    "portfolio_expected_loss": float(losses.to_numpy().mean(axis=0) @ weights),
    "portfolio_standard_deviation": float(portfolio_std),
}


### Interpretation

Diversification benefits depend on covariance, not only on individual sector risks. The model still requires validated distributions, scenario stress tests, tail-risk analysis, and institutional judgment.


## 18. Engineering notes

- Covariance matrices may be poorly conditioned.
- Correlation can hide nonlinear dependence.
- Sample moments are sensitive to outliers.
- Concentration bounds require specific assumptions.
- Tail-risk decisions should not rely only on variance.
- Numerical precision does not compensate for weak data.


## 19. Common errors

- Interpreting expectation as a guaranteed outcome.
- Confusing variance with standard deviation.
- Treating correlation as causation.
- Ignoring covariance in aggregate risk.
- Applying concentration bounds without checking assumptions.
- Reporting only central moments for heavy-tailed risks.


## 20. Exercises

### Level A
Explain expectation, variance, covariance, and correlation.

### Level B
Derive \(\operatorname{Var}(aX+b)\).

### Level C
Simulate empirical tail probabilities and compare them with Chebyshev and Hoeffding bounds.

### Capstone
Construct a multi-sector risk portfolio, estimate its covariance structure, compute aggregate uncertainty, and stress-test dependence assumptions.


## 21. Key insight

Expectation measures center, variance measures spread, covariance measures shared movement, and concentration bounds quantify deviation risk. Together, they form a core language for uncertainty, statistics, AI, and Decision Intelligence.
